# Data Pre-processing Experimentation

In [72]:
import nibabel as nib
import numpy as np
from pathlib import Path
import pandas as pd
import re

In [73]:
class DataSubject:
    def __init__(self, cohort, patient_id, t1, t1c, t2, flair, age, sex, label):
        self.cohort = cohort
        self.patient_id = patient_id

        self.t1 = t1
        self.t1c = t1c
        self.t2 = t2
        self.flair = flair

        self.age = age
        self.sex = sex
        self.label = label

    def load(self, sequence):
        """Load one sequence as a numpy array, e.g. subject.load("flair")."""
        path = getattr(self, sequence)
        return np.asarray(nib.load(path).dataobj, dtype=np.float32)

    def load_nifti(self, sequence):
        """Load one sequence as a nibabel image"""
        return nib.load(getattr(self, sequence))

## UCSF Cohort

In [74]:
df = pd.read_csv("../../data/UCSF-PDGM/UCSF-PDGM-metadata_v5.csv")
df = df[["ID", "Sex", "Age at MRI", "IDH"]]
ucsf_df = df[~df['ID'].str.contains(r'FU\d{3}d', regex=True, na=False)]
ucsf_df

,ID,Sex,Age at MRI,IDH
0,UCSF-PDGM-004,M,66,wildtype
1,UCSF-PDGM-005,F,80,wildtype
2,UCSF-PDGM-007,M,70,wildtype
3,UCSF-PDGM-008,M,70,wildtype
4,UCSF-PDGM-009,F,68,wildtype
...,...,...,...,...
496,UCSF-PDGM-537,M,64,wildtype
497,UCSF-PDGM-538,F,41,wildtype
498,UCSF-PDGM-539,M,72,wildtype
499,UCSF-PDGM-540,F,70,IDH1 p.R132H


In [75]:
root = Path("../../data/UCSF-PDGM/PKG - UCSF-PDGM Version 5/UCSF-PDGM-v5/")

SEQUENCES = [
    "T1",
    "T1c",
    "T2",
    "FLAIR"
]

def folder_id(csv_id, width=4):
    num = int(re.search(r"(\d+)$", csv_id).group(1))
    return f"UCSF-PDGM-{num:0{width}d}"

def find_sequence(csv_id, suffix):
    fid = folder_id(csv_id)
    p = root / f"{fid}_nifti" / f"{fid}_{suffix}_bias.nii.gz"
    return p if p.exists() else None

ucsf_df = ucsf_df.copy()
ucsf_df["patient_id"] = ucsf_df["ID"].apply(folder_id)   # canonical 4-digit ID

for seq in SEQUENCES:
    ucsf_df[seq] = ucsf_df["ID"].apply(lambda pid: find_sequence(pid, seq))

complete = ucsf_df.notna().all(axis=1)
print(complete.all())

True


In [76]:
ucsf_subjects = []

for index, row in ucsf_df.iterrows():
    ucsf_subjects.append(
        DataSubject(
            cohort = "UCSF-PDGM",
            patient_id = row["ID"],
            t1 = row["T1"],
            t1c = row["T1c"],
            t2 = row["T2"],
            flair = row["FLAIR"],
            age = row["Age at MRI"],
            sex = row["Sex"],
            label = row["IDH"]
        )
    )

print(len(ucsf_subjects))

495


## UTSW Cohort

In [94]:
df = pd.read_csv("../../data/UTSW-Glioma/UTSW_Glioma_Metadata.tsv", sep="\t")
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH", "T1", "T1GD", "T2", "T2FLAIR"]]

SEQUENCES = ["T1", "T1GD", "T2", "T2FLAIR"]
df = df[df[SEQUENCES].eq("Available").all(axis=1)]
df = df.dropna(subset=["IDH"])
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH"]]
df

,Subject ID,Sex at birth,Age at Imaging,IDH
0,BT0001,F,44,wild type
1,BT0002,M,58,wild type
2,BT0003,F,22,mutated
3,BT0005,M,67,wild type
4,BT0007,M,30,mutated
...,...,...,...,...
620,BT1300,M,59,mutated
621,BT1301,F,20,mutated
622,BT1303,F,42,mutated
623,BT1311,M,50,wild type


In [95]:
root = Path("../../data/UTSW-Glioma/PKG - UTSW-Glioma/UTSW-Glioma/")

SEQUENCES = [
    "t1", "t1ce", "t2", "flair"
]

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"brain_{suffix}.nii.gz"
    return p if p.exists() else None

utsw_df = df.copy()
for seq in SEQUENCES:
    utsw_df[seq] = utsw_df["Subject ID"].apply(lambda pid: find_sequence(pid, seq))

complete = utsw_df.notna().all(axis=1)
print(complete.all())

True


In [98]:
utsw_subjects = []

for index, row in utsw_df.iterrows():
    utsw_subjects.append(
        DataSubject(
            cohort = "UTSW-Glioma",
            patient_id = row["Subject ID"],
            t1 = row["t1"],
            t1c = row["t1ce"],
            t2 = row["t2"],
            flair = row["flair"],
            age = row["Age at Imaging"],
            sex = row["Sex at birth"],
            label = row["IDH"]
        )
    )

print(len(utsw_subjects))

622


## EGD Cohort

In [118]:
df = pd.read_csv("../../data/EGD/egd_data/metadata/subjects.csv")
df = df[["subject_label", "fields/field[0]/field", "demographics/gender", "demographics/age"]]
df = df.rename(columns={"fields/field[0]/field":"IDH", "demographics/gender":"gender", "demographics/age":"age"})
df

,subject_label,IDH,gender,age
0,EGD-0001,-1.0,male,77.0
1,EGD-0002,-1.0,male,48.0
2,EGD-0003,-1.0,female,32.0
3,EGD-0004,1.0,female,45.0
4,EGD-0005,-1.0,female,39.0
...,...,...,...,...
775,EGD-2570,NaN,NaN,NaN
776,EGD-2577,NaN,NaN,NaN
777,EGD-2587,NaN,NaN,NaN
778,EGD-2615,NaN,NaN,NaN


In [119]:
n0 = len(df)

df = df.replace([-1, "-1"], np.nan)
df = df.replace(r"(?i)^\s*unknown\s*$", np.nan, regex=True)
df = df.dropna().reset_index(drop=True)

print(f"{n0} -> {len(df)} rows")

780 -> 424 rows


In [ ]:
root = Path("../../data/EGD/egd_data/images/")

SEQUENCES = [
    "t1", "t1ce", "t2", "flair"
]

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"brain_{suffix}.nii.gz"
    return p if p.exists() else None

utsw_df = df.copy()
for seq in SEQUENCES:
    utsw_df[seq] = utsw_df["Subject ID"].apply(lambda pid: find_sequence(pid, seq))

complete = utsw_df.notna().all(axis=1)
print(complete.all())

# Data Pre-processing Experimentation

In [72]:
import nibabel as nib
import numpy as np
from pathlib import Path
import pandas as pd
import re

In [73]:
class DataSubject:
    def __init__(self, cohort, patient_id, t1, t1c, t2, flair, age, sex, label):
        self.cohort = cohort
        self.patient_id = patient_id

        self.t1 = t1
        self.t1c = t1c
        self.t2 = t2
        self.flair = flair

        self.age = age
        self.sex = sex
        self.label = label

    def load(self, sequence):
        """Load one sequence as a numpy array, e.g. subject.load("flair")."""
        path = getattr(self, sequence)
        return np.asarray(nib.load(path).dataobj, dtype=np.float32)

    def load_nifti(self, sequence):
        """Load one sequence as a nibabel image"""
        return nib.load(getattr(self, sequence))

## UCSF Cohort

In [74]:
df = pd.read_csv("../../data/UCSF-PDGM/UCSF-PDGM-metadata_v5.csv")
df = df[["ID", "Sex", "Age at MRI", "IDH"]]
ucsf_df = df[~df['ID'].str.contains(r'FU\d{3}d', regex=True, na=False)]
ucsf_df

,ID,Sex,Age at MRI,IDH
0,UCSF-PDGM-004,M,66,wildtype
1,UCSF-PDGM-005,F,80,wildtype
2,UCSF-PDGM-007,M,70,wildtype
3,UCSF-PDGM-008,M,70,wildtype
4,UCSF-PDGM-009,F,68,wildtype
...,...,...,...,...
496,UCSF-PDGM-537,M,64,wildtype
497,UCSF-PDGM-538,F,41,wildtype
498,UCSF-PDGM-539,M,72,wildtype
499,UCSF-PDGM-540,F,70,IDH1 p.R132H


In [75]:
root = Path("../../data/UCSF-PDGM/PKG - UCSF-PDGM Version 5/UCSF-PDGM-v5/")

SEQUENCES = [
    "T1",
    "T1c",
    "T2",
    "FLAIR"
]

def folder_id(csv_id, width=4):
    num = int(re.search(r"(\d+)$", csv_id).group(1))
    return f"UCSF-PDGM-{num:0{width}d}"

def find_sequence(csv_id, suffix):
    fid = folder_id(csv_id)
    p = root / f"{fid}_nifti" / f"{fid}_{suffix}_bias.nii.gz"
    return p if p.exists() else None

ucsf_df = ucsf_df.copy()
ucsf_df["patient_id"] = ucsf_df["ID"].apply(folder_id)   # canonical 4-digit ID

for seq in SEQUENCES:
    ucsf_df[seq] = ucsf_df["ID"].apply(lambda pid: find_sequence(pid, seq))

complete = ucsf_df.notna().all(axis=1)
print(complete.all())

True


In [76]:
ucsf_subjects = []

for index, row in ucsf_df.iterrows():
    ucsf_subjects.append(
        DataSubject(
            cohort = "UCSF-PDGM",
            patient_id = row["ID"],
            t1 = row["T1"],
            t1c = row["T1c"],
            t2 = row["T2"],
            flair = row["FLAIR"],
            age = row["Age at MRI"],
            sex = row["Sex"],
            label = row["IDH"]
        )
    )

print(len(ucsf_subjects))

495


## UTSW Cohort

In [94]:
df = pd.read_csv("../../data/UTSW-Glioma/UTSW_Glioma_Metadata.tsv", sep="\t")
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH", "T1", "T1GD", "T2", "T2FLAIR"]]

SEQUENCES = ["T1", "T1GD", "T2", "T2FLAIR"]
df = df[df[SEQUENCES].eq("Available").all(axis=1)]
df = df.dropna(subset=["IDH"])
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH"]]
df

,Subject ID,Sex at birth,Age at Imaging,IDH
0,BT0001,F,44,wild type
1,BT0002,M,58,wild type
2,BT0003,F,22,mutated
3,BT0005,M,67,wild type
4,BT0007,M,30,mutated
...,...,...,...,...
620,BT1300,M,59,mutated
621,BT1301,F,20,mutated
622,BT1303,F,42,mutated
623,BT1311,M,50,wild type


In [95]:
root = Path("../../data/UTSW-Glioma/PKG - UTSW-Glioma/UTSW-Glioma/")

SEQUENCES = [
    "t1", "t1ce", "t2", "flair"
]

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"brain_{suffix}.nii.gz"
    return p if p.exists() else None

utsw_df = df.copy()
for seq in SEQUENCES:
    utsw_df[seq] = utsw_df["Subject ID"].apply(lambda pid: find_sequence(pid, seq))

complete = utsw_df.notna().all(axis=1)
print(complete.all())

True


In [98]:
utsw_subjects = []

for index, row in utsw_df.iterrows():
    utsw_subjects.append(
        DataSubject(
            cohort = "UTSW-Glioma",
            patient_id = row["Subject ID"],
            t1 = row["t1"],
            t1c = row["t1ce"],
            t2 = row["t2"],
            flair = row["flair"],
            age = row["Age at Imaging"],
            sex = row["Sex at birth"],
            label = row["IDH"]
        )
    )

print(len(utsw_subjects))

622


## EGD Cohort

In [118]:
df = pd.read_csv("../../data/EGD/egd_data/metadata/subjects.csv")
df = df[["subject_label", "fields/field[0]/field", "demographics/gender", "demographics/age"]]
df = df.rename(columns={"fields/field[0]/field":"IDH", "demographics/gender":"gender", "demographics/age":"age"})
df

,subject_label,IDH,gender,age
0,EGD-0001,-1.0,male,77.0
1,EGD-0002,-1.0,male,48.0
2,EGD-0003,-1.0,female,32.0
3,EGD-0004,1.0,female,45.0
4,EGD-0005,-1.0,female,39.0
...,...,...,...,...
775,EGD-2570,NaN,NaN,NaN
776,EGD-2577,NaN,NaN,NaN
777,EGD-2587,NaN,NaN,NaN
778,EGD-2615,NaN,NaN,NaN


In [119]:
n0 = len(df)

df = df.replace([-1, "-1"], np.nan)
df = df.replace(r"(?i)^\s*unknown\s*$", np.nan, regex=True)
df = df.dropna().reset_index(drop=True)

print(f"{n0} -> {len(df)} rows")

780 -> 424 rows


In [120]:
root = Path("../../data/EGD/egd_data/images/")

SEQUENCES = {
    "T1" : 1,
    "T1GD": 2,
    "T2": 3,
    "FLAIR": 4
}

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"MR_{patient_id}" / f"scan{SEQUENCES[suffix]}_{suffix}_{suffix}" / "NIFTI" / f"{suffix}.nii.gz"
    return p if p.exists() else None

egd_df = df.copy()
for seq in SEQUENCES.keys():
    egd_df[seq] = egd_df["subject_label"].apply(lambda pid: find_sequence(pid, seq))

complete = egd_df.notna().all(axis=1)
print(complete.all())

True


In [121]:
egd_df

,subject_label,IDH,gender,age,T1,T1GD,T2,FLAIR
0,EGD-0004,1.0,female,45.0,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...
1,EGD-0008,1.0,female,60.0,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...
2,EGD-0009,0.0,male,45.0,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...
3,EGD-0011,1.0,female,49.0,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...
4,EGD-0014,0.0,male,84.0,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...
...,...,...,...,...,...,...,...,...
419,EGD-0765,0.0,male,57.0,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...
420,EGD-0768,0.0,male,56.0,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...
421,EGD-0771,1.0,female,38.0,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...
422,EGD-0772,0.0,female,64.0,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...


In [123]:
egd_subjects = []

for index, row in egd_df.iterrows():
    egd_subjects.append(
        DataSubject(
            cohort = "EGD",
            patient_id = row["subject_label"],
            t1 = row["T1"],
            t1c = row["T1GD"],
            t2 = row["T2"],
            flair = row["FLAIR"],
            age = row["age"],
            sex = row["gender"],
            label = row["IDH"]
        )
    )

print(len(egd_subjects))

424
